## Training Log: Why the Initial Forwards-Only Experiments Performed Poorly

This notebook tested several simple modelling approaches on 2025/26 forwards only. Overall, the results were poor because the dataset was too small, sparse, and noisy for predicting next-gameweek FPL returns directly. Most player-gameweek rows had 0 or very low points, while high-scoring returns were rare. As a result, most models learned conservative low predictions. This made the overall MAE/RMSE look somewhat acceptable, but the models failed badly on the most important FPL cases: predicting hauls.

The main target, `total_points`, is also very noisy. It combines many different events: minutes, goals, assists, bonus, cards, appearance points, and other scoring rules. A player can have strong underlying attacking stats and still blank, while another player can score from one chance and haul. Because of this, directly predicting total FPL points from one season of forwards only is too unstable.

## Why Each Method Failed

### 1. Basic direct `total_points` regression

The first main approach trained a LightGBM regressor to predict next-gameweek `total_points` using rolling historical features. This performed poorly because most rows had low or zero points, so the model learned to predict low values for almost everyone. Since hauls are rare, predicting conservatively is rewarded by the overall loss function. However, this means the model badly underpredicts high scorers. For example, actual scores like 10, 12, or 16 points were often predicted as only around 1 to 3 points.

This method failed because it treated FPL points as one direct target instead of breaking them into football components such as minutes, goals, assists, and bonus.

### 2. High-minutes-only regression

The second approach filtered to players with higher recent average minutes before training. The idea was reasonable because low-minute bench players create a lot of noisy 0-point rows. However, this made the dataset much smaller. After filtering, there were too few rows left for the model to learn meaningful patterns, especially with many features.

The model effectively collapsed into predicting a near-constant value around 2 points. This happened because the filtered dataset was too small, and the model could not confidently find useful splits. So although filtering removed some noise, it also removed too much data.

### 3. Two-stage haul model

The two-stage method first tried to classify whether a player would haul, then adjusted the regression prediction using the haul probability. This was a good idea in principle because hauls are especially important in FPL. However, it still failed because there were very few positive haul examples in the training data. The classifier had limited examples to learn from, and the final prediction adjustment was too small to meaningfully lift predictions for true high scorers.

Even if the model detected that a player had a higher chance of hauling, the final prediction still usually stayed low. For example, adding a small haul-probability adjustment to a 2-point prediction does not turn it into a realistic 8 to 12 point prediction. So the method slightly improved awareness of haul risk, but did not solve the core problem.

### 4. Rolling-sum feature approach

The notebook used rolling sums over the past 3 or 5 gameweeks for many features. This is a decent starting point, but it mixes playing time and player quality together. For example, a player with 1.0 xG over 450 minutes and a player with 1.0 xG over 120 minutes look similar if only the rolling xG sum is used, even though their attacking rate is very different.

This approach also makes recent minutes dominate many features. A player may look weak simply because he played fewer minutes, not because he was less dangerous when on the pitch. For FPL modelling, rolling sums should be combined with per-90 features such as xG per 90, xA per 90, shots per 90, big chances per 90, and BPS per 90.

### 5. Forwards-only training

Training only on forwards made the dataset much smaller. This limited the number of examples the model could learn from, especially for rare events like goals, assists, and hauls. It also made the model less generalisable.

A better approach is likely to train component models using all positions where sensible, with `position` included as a feature. For example, goals and assists can be predicted across defenders, midfielders, and forwards, then converted into FPL points using position-specific scoring rules.

## Other Data and Design Issues

The `name` column appeared to represent the team name rather than the player name, which made debugging harder. The model also excluded `player_id`, so it could not directly learn player-specific tendencies such as penalty-taking, nailedness, finishing ability, or bonus potential. Many team-level match stats were repeated for all players on the same team, which may help capture team strength but does not fully represent individual player quality.

## Future Improvements

A better modelling approach is to avoid predicting `total_points` directly at first. Instead, train separate component models for key FPL scoring parts:

- expected minutes
- expected goals or xG
- expected assists or xA
- clean sheet probability
- expected bonus or BPS
- card risk later if needed

These predicted components can then be converted into expected FPL points using the official FPL scoring rules. This is more interpretable and less noisy than direct points prediction.

Future versions should also use multiple seasons, include all positions where appropriate, add `position` and player identity features, and create stronger per-90 rolling features. Evaluation should focus not only on overall MAE/RMSE, but also on FPL-relevant metrics such as performance on high-minute players, top-k predicted players per gameweek, and how well the model identifies potential hauls.

In [3]:
# === BUILD FWD TRAINING DF FROM CSV (GW6+, rolling sums 3/5) ===
import pandas as pd
import numpy as np
from pathlib import Path

# Load source CSV (df_fwd_att)
CWD = Path.cwd().resolve()
BASE = CWD / 'data' / 'intermediate'
if not BASE.exists():
    BASE = CWD.parent / 'data' / 'intermediate'

src_path = BASE / '2025-2026_df_fwd_att.csv'
df = pd.read_csv(src_path)

# Basic cleanup / ordering
df['gameweek'] = pd.to_numeric(df['gameweek'], errors='coerce')
df = df.sort_values(['player_id', 'gameweek']).reset_index(drop=True)

# Columns known beforehand (keep as-is, no rolling transform)
known_pre_match_cols = [
    'player_id',
    'position',
    'name',
    'side',
    'team_elo',
    'gameweek'
]

# Target
target_col = 'total_points'

# Numeric columns to engineer rolling features for
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_from_roll = set(known_pre_match_cols + [target_col])
roll_base_cols = [c for c in num_cols if c not in exclude_from_roll]

# Create lagged rolling sums per player (excluding current GW)
g = df.groupby('player_id', group_keys=False)
for c in roll_base_cols:
    lag = g[c].shift(1)
    df[f'{c}_sum_last3'] = lag.rolling(window=3, min_periods=3).sum()
    df[f'{c}_sum_last5'] = lag.rolling(window=5, min_periods=5).sum()

# Past points form (leakage-safe due to shift)
lag_pts = g[target_col].shift(1)
df['total_points_sum_last3'] = lag_pts.rolling(window=3, min_periods=3).sum()
df['total_points_sum_last5'] = lag_pts.rolling(window=5, min_periods=5).sum()

# Keep only GW6 onward
df = df[df['gameweek'] >= 6].copy()

# Final training dataframe
engineered_cols = [c for c in df.columns if c.endswith('_sum_last3') or c.endswith('_sum_last5')]
final_cols = [c for c in known_pre_match_cols if c in df.columns] + [target_col] + engineered_cols
df_fwd_train = df[final_cols].copy()

# Keep only fully complete rows
df_fwd_train = df_fwd_train.dropna().reset_index(drop=True)

print('Loaded:', src_path)
print('Base rows:', len(df))
print('Training rows (GW6+, complete only):', len(df_fwd_train))
print('Training shape:', df_fwd_train.shape)
df_fwd_train.head()


Loaded: /mnt/c/Users/easan.s.2024/Documents/GitHub/FPL-Oracle/data/intermediate/2025-2026_df_fwd_att.csv
Base rows: 2618
Training rows (GW6+, complete only): 2209
Training shape: (2209, 75)


/tmp/ipykernel_32902/3110125367.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{c}_sum_last5'] = lag.rolling(window=5, min_periods=5).sum()
/tmp/ipykernel_32902/3110125367.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{c}_sum_last3'] = lag.rolling(window=3, min_periods=3).sum()
/tmp/ipykernel_32902/3110125367.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.

,player_id,position,name,side,team_elo,gameweek,total_points,minutes_sum_last3,minutes_sum_last5,goals_scored_sum_last3,...,passes_sum_last3,passes_sum_last5,touches_in_opposition_box_sum_last3,touches_in_opposition_box_sum_last5,successful_dribbles_sum_last3,successful_dribbles_sum_last5,successful_dribbles_pct_sum_last3,successful_dribbles_pct_sum_last5,total_points_sum_last3,total_points_sum_last5
0,30,Forward,Arsenal,away,2038.875,6,0.0,0.0,30.0,0.0,...,1144.0,2037.0,86.0,140.0,14.5,35.5,120.5,206.5,0.0,1.0
1,30,Forward,Arsenal,home,2015.990,7,0.0,0.0,0.0,0.0,...,1174.0,2144.0,87.5,150.5,20.0,35.0,140.0,228.0,0.0,0.0
2,30,Forward,Arsenal,away,2018.120,8,0.0,0.0,0.0,0.0,...,1349.5,2111.0,103.5,162.5,17.5,28.0,134.0,216.0,0.0,0.0
3,30,Forward,Arsenal,away,2046.110,11,0.0,0.0,0.0,0.0,...,1412.5,2379.5,92.5,169.0,20.0,33.5,157.0,252.5,0.0,0.0
4,30,Forward,Arsenal,away,2037.350,13,0.0,0.0,0.0,0.0,...,1395.5,2432.0,75.0,145.0,17.5,34.0,130.5,250.5,0.0,0.0


In [3]:
%pip install -q lightgbm xgboost scikit-learn pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
  import lightgbm, xgboost, sklearn, pandas, numpy
  print("OK imports")
  print("lightgbm:", lightgbm.__version__)
  print("xgboost:", xgboost.__version__)
  print("sklearn:", sklearn.__version__)

OK imports
lightgbm: 4.6.0
xgboost: 3.2.0
sklearn: 1.8.0


In [4]:
# === TIME-BASED TRAIN/VAL/TEST SPLIT ===
# expects df_fwd_train already created
df_split = df_fwd_train.copy()
df_split['gameweek'] = pd.to_numeric(df_split['gameweek'], errors='coerce')
df_split = df_split.dropna(subset=['gameweek']).sort_values('gameweek').reset_index(drop=True)

# Split by unique gameweeks (time-based)
gws = sorted(df_split['gameweek'].unique())
n_gw = len(gws)

train_end = int(n_gw * 0.70)
val_end = int(n_gw * 0.85)

train_gws = gws[:train_end]
val_gws = gws[train_end:val_end]
test_gws = gws[val_end:]

df_train = df_split[df_split['gameweek'].isin(train_gws)].copy()
df_val = df_split[df_split['gameweek'].isin(val_gws)].copy()
df_test = df_split[df_split['gameweek'].isin(test_gws)].copy()

target_col = 'total_points'
id_cols = ['player_id', 'position', 'name', 'side', 'gameweek']
feature_cols = [c for c in df_split.columns if c not in id_cols + [target_col]]

X_train, y_train = df_train[feature_cols], df_train[target_col]
X_val, y_val = df_val[feature_cols], df_val[target_col]
X_test, y_test = df_test[feature_cols], df_test[target_col]

print('Train GWs:', (min(train_gws), max(train_gws)), '| rows:', len(df_train))
print('Val   GWs:', (min(val_gws), max(val_gws)), '| rows:', len(df_val))
print('Test  GWs:', (min(test_gws), max(test_gws)), '| rows:', len(df_test))
print('Features:', len(feature_cols))


Train GWs: (np.int64(6), np.int64(26)) | rows: 1579
Val   GWs: (np.int64(27), np.int64(30)) | rows: 331
Test  GWs: (np.int64(31), np.int64(35)) | rows: 299
Features: 69


In [5]:
# === TRAIN FORWARDS MODEL (LIGHTGBM + EARLY STOPPING) ===
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Requires: X_train, y_train, X_val, y_val (from split cell)

use_lgbm = True
try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
except Exception:
    use_lgbm = False

if use_lgbm:
    model = LGBMRegressor(
        objective='regression_l1',
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=25,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='l1',
        callbacks=[
            early_stopping(stopping_rounds=150, verbose=True),
            log_evaluation(period=100)
        ]
    )
else:
    from sklearn.ensemble import HistGradientBoostingRegressor
    model = HistGradientBoostingRegressor(
        loss='absolute_error',
        learning_rate=0.03,
        max_depth=6,
        max_iter=800,
        min_samples_leaf=25,
        random_state=42,
        verbose=1
    )
    model.fit(X_train, y_train)

# Validation metrics (part of training workflow)
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

y_val_pred = model.predict(X_val)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = rmse(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

print('Model:', type(model).__name__)
if hasattr(model, 'best_iteration_') and model.best_iteration_ is not None:
    print('Best iteration:', model.best_iteration_)
print(f'VAL | MAE: {val_mae:.4f} | RMSE: {val_rmse:.4f} | R2: {val_r2:.4f}')

# Top features (if available)
if hasattr(model, 'feature_importances_'):
    imp = np.array(model.feature_importances_, dtype=float)
    order = np.argsort(imp)[::-1][:20]
    print('\nTop 20 feature importances:')
    for i in order:
        print(f'  {feature_cols[i]}: {imp[i]:.6f}')


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.213749 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8156
[LightGBM] [Info] Number of data points in the train set: 1579, number of used features: 64
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 150 rounds
[100]	valid_0's l1: 0.891567
[200]	valid_0's l1: 0.86045
[300]	valid_0's l1: 0.871708
Early stopping, best iteration is:
[185]	valid_0's l1: 0.859871
Model: LGBMRegressor
Best iteration: 185
VAL | MAE: 0.8599 | RMSE: 2.1521 | R2: 0.2256

Top 20 feature importances:
  minutes_sum_last3: 338.000000
  minutes_sum_last5: 246.000000
  big_chances_sum_last5: 186.000000
  successful_dribbles_pct_sum_last5: 168.000000
  xg_on_target_xgot_sum_last5: 155.000000
  bps_sum_last3: 155.000000
  bps_sum_last5: 155.000000

In [6]:
# === TEST EVALUATION (SEPARATE) ===
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Requires: fitted `model`, and X_test/y_test from split cell

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

y_test_pred = model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = rmse(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f'TEST | MAE: {test_mae:.4f} | RMSE: {test_rmse:.4f} | R2: {test_r2:.4f}')


TEST | MAE: 1.0900 | RMSE: 2.6698 | R2: 0.1618


In [8]:
# === SAMPLE 5 RANDOM TEST ROWS: PRED VS ACTUAL ===
import numpy as np
import pandas as pd

# Requires: model, df_test, X_test, y_test

test_df = df_test.copy().reset_index(drop=True)
X_test_local = X_test.reset_index(drop=True)
y_test_local = y_test.reset_index(drop=True)

sample_n = 5
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(test_df), size=min(sample_n, len(test_df)), replace=False)

X_sample = X_test_local.iloc[sample_idx]
y_true_sample = y_test_local.iloc[sample_idx]
y_pred_sample = model.predict(X_sample)

out = test_df.iloc[sample_idx][['player_id', 'name', 'position', 'side', 'gameweek']].copy()
out['actual_points'] = y_true_sample.values
out['pred_points'] = y_pred_sample
out['abs_error'] = (out['actual_points'] - out['pred_points']).abs()

print(out.sort_values('gameweek').to_string(index=False))


 player_id          name position side  gameweek  actual_points  pred_points  abs_error
       250       Chelsea  Forward away        31            1.0     1.017867   0.017867
       365         Leeds  Forward away        32            0.0     0.766159   0.766159
       338        Fulham  Forward away        32            2.0     1.114458   0.885542
       805 Nott'm Forest  Forward away        34            0.0     0.000000   0.000000
       311       Everton  Forward away        34            0.0     2.686437   2.686437


In [9]:
# === SAMPLE 5 RANDOM TEST ROWS (ACTUAL POINTS > 6) ===
import numpy as np
import pandas as pd

# Requires: model, df_test, X_test, y_test

test_df = df_test.copy().reset_index(drop=True)
X_test_local = X_test.reset_index(drop=True)
y_test_local = y_test.reset_index(drop=True)

eligible_idx = y_test_local[y_test_local > 6].index.to_numpy()
print('Eligible test rows with actual_points > 6:', len(eligible_idx))

if len(eligible_idx) == 0:
    print('No test rows found with actual points > 6.')
else:
    sample_n = min(5, len(eligible_idx))
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(eligible_idx, size=sample_n, replace=False)

    X_sample = X_test_local.iloc[sample_idx]
    y_true_sample = y_test_local.iloc[sample_idx]
    y_pred_sample = model.predict(X_sample)

    out = test_df.iloc[sample_idx][['player_id', 'name', 'position', 'side', 'gameweek']].copy()
    out['actual_points'] = y_true_sample.values
    out['pred_points'] = y_pred_sample
    out['abs_error'] = (out['actual_points'] - out['pred_points']).abs()

    print(out.sort_values('gameweek').to_string(index=False))


Eligible test rows with actual_points > 6: 21
 player_id      name position side  gameweek  actual_points  pred_points  abs_error
       311   Everton  Forward home        31           16.0     1.925166  14.074834
       500 Newcastle  Forward away        32            8.0     0.596668   7.403332
       500 Newcastle  Forward home        33            9.0     1.621244   7.378756
       499 Liverpool  Forward home        34            8.0     1.313574   6.686426
       624  West Ham  Forward home        34           10.0     2.315405   7.684595


In [10]:
# === TWO-STAGE TRAINING: HAUL CLASSIFIER + WEIGHTED REGRESSOR ===
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score

# Requires: X_train, y_train, X_val, y_val, X_test, y_test
# Stage 1: classify haul (points > 6)
y_train_haul = (y_train > 6).astype(int)
y_val_haul = (y_val > 6).astype(int)

try:
    from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping, log_evaluation

    cls = LGBMClassifier(
        objective='binary',
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    cls.fit(
        X_train, y_train_haul,
        eval_set=[(X_val, y_val_haul)],
        eval_metric='auc',
        callbacks=[early_stopping(100, verbose=True), log_evaluation(100)]
    )

    # Stage 2: weighted regressor for points (upweight high-point rows)
    w_train = np.where(y_train > 6, 3.0, 1.0)
    reg = LGBMRegressor(
        objective='regression_l1',
        n_estimators=2500,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.5,
        random_state=42,
        n_jobs=-1
    )
    reg.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_val, y_val)],
        eval_metric='l1',
        callbacks=[early_stopping(120, verbose=True), log_evaluation(100)]
    )
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
    cls = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=400, random_state=42)
    cls.fit(X_train, y_train_haul)
    reg = HistGradientBoostingRegressor(loss='absolute_error', max_depth=6, learning_rate=0.03, max_iter=800, random_state=42)
    reg.fit(X_train, y_train)

p_haul_val = cls.predict_proba(X_val)[:, 1]
p_haul_test = cls.predict_proba(X_test)[:, 1]

pred_reg_val = reg.predict(X_val)
pred_reg_test = reg.predict(X_test)

# Blend to lift upside without exploding predictions
alpha = 1.25
pred_val_2stage = np.clip(pred_reg_val + alpha * p_haul_val, 0, None)
pred_test_2stage = np.clip(pred_reg_test + alpha * p_haul_test, 0, None)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print('Classifier AUC (val):', round(roc_auc_score(y_val_haul, p_haul_val), 4))
print('2-stage VAL  | MAE:', round(mean_absolute_error(y_val, pred_val_2stage),4),
      '| RMSE:', round(rmse(y_val, pred_val_2stage),4),
      '| R2:', round(r2_score(y_val, pred_val_2stage),4))
print('2-stage TEST | MAE:', round(mean_absolute_error(y_test, pred_test_2stage),4),
      '| RMSE:', round(rmse(y_test, pred_test_2stage),4),
      '| R2:', round(r2_score(y_test, pred_test_2stage),4))


[LightGBM] [Info] Number of positive: 84, number of negative: 1495
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.246036 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8156
[LightGBM] [Info] Number of data points in the train set: 1579, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.053198 -> initscore=-2.879065
[LightGBM] [Info] Start training from score -2.879065
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.818069	valid_0's binary_logloss: 0.183547
Early stopping, best iteration is:
[81]	valid_0's auc: 0.821441	valid_0's binary_logloss: 0.181616
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013227 seconds.
You can set `force_col_wise=true` to remov

In [11]:
# === DIAGNOSTIC: HIGH-POINT ROWS PERFORMANCE (>6) ===
import numpy as np
import pandas as pd

mask_hi = y_test > 6
n_hi = int(mask_hi.sum())
print('High-point test rows (>6):', n_hi)

if n_hi > 0:
    mae_hi = np.mean(np.abs(y_test[mask_hi] - pred_test_2stage[mask_hi]))
    print('2-stage MAE on >6 rows:', round(float(mae_hi), 4))

    # sample 5 high-point rows
    tmp = df_test.reset_index(drop=True).copy()
    yy = y_test.reset_index(drop=True)
    pp = pd.Series(pred_test_2stage).reset_index(drop=True)
    hi_idx = yy[yy > 6].index.to_numpy()
    sel = np.random.default_rng(42).choice(hi_idx, size=min(5, len(hi_idx)), replace=False)

    out = tmp.loc[sel, ['player_id','name','position','side','gameweek']].copy()
    out['actual_points'] = yy.loc[sel].values
    out['pred_points_2stage'] = pp.loc[sel].values
    out['abs_error'] = (out['actual_points'] - out['pred_points_2stage']).abs()
    print(out.sort_values('gameweek').to_string(index=False))
else:
    print('No >6 rows in test set.')


High-point test rows (>6): 21
2-stage MAE on >6 rows: 8.6839
 player_id      name position side  gameweek  actual_points  pred_points_2stage  abs_error
       311   Everton  Forward home        31           16.0            2.564596  13.435404
       500 Newcastle  Forward away        32            8.0            0.607069   7.392931
       500 Newcastle  Forward home        33            9.0            1.855913   7.144087
       499 Liverpool  Forward home        34            8.0            0.927498   7.072502
       624  West Ham  Forward home        34           10.0            2.832814   7.167186


In [12]:
# === HIGH-MINUTES FILTER + SPLIT (FORWARDS) ===
import pandas as pd
import numpy as np

# Rebuild from CSV to be self-contained
CWD = Path.cwd().resolve()
BASE = CWD / "data" / "intermediate"
if not BASE.exists():
    BASE = CWD.parent / "data" / "intermediate"

src_path = BASE / "2025-2026_df_fwd_att.csv"
df = pd.read_csv(src_path)

df["gameweek"] = pd.to_numeric(df["gameweek"], errors="coerce")
df = df.sort_values(["player_id", "gameweek"]).reset_index(drop=True)

known_pre_match_cols = ["player_id", "position", "name", "side", "team_elo", "gameweek"]
target_col = "total_points"

# Rolling sums over lagged features
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_from_roll = set(known_pre_match_cols + [target_col])
roll_base_cols = [c for c in num_cols if c not in exclude_from_roll]
g = df.groupby("player_id", group_keys=False)
for c in roll_base_cols:
    lag = g[c].shift(1)
    df[f"{c}_sum_last3"] = lag.rolling(window=3, min_periods=3).sum()
    df[f"{c}_sum_last5"] = lag.rolling(window=5, min_periods=5).sum()

lag_pts = g[target_col].shift(1)
df["total_points_sum_last3"] = lag_pts.rolling(window=3, min_periods=3).sum()
df["total_points_sum_last5"] = lag_pts.rolling(window=5, min_periods=5).sum()

# GW6+
df = df[df["gameweek"] >= 6].copy()

# High-minutes eligibility using HISTORY only
# Condition: average minutes over last 5 >= 45
if "minutes_sum_last5" not in df.columns:
    raise RuntimeError("minutes_sum_last5 not found; ensure minutes column exists in source CSV.")

df["minutes_avg_last5"] = df["minutes_sum_last5"] / 5.0
df_hi = df[df["minutes_avg_last5"] >= 45].copy()

# Keep only complete rows
engineered_cols = [c for c in df_hi.columns if c.endswith("_sum_last3") or c.endswith("_sum_last5")]
final_cols = [c for c in known_pre_match_cols if c in df_hi.columns] + [target_col, "minutes_avg_last5"] + engineered_cols
df_fwd_train_hi = df_hi[final_cols].dropna().reset_index(drop=True)

# Time split
gws = sorted(df_fwd_train_hi["gameweek"].unique())
n_gw = len(gws)
train_end = int(n_gw * 0.70)
val_end = int(n_gw * 0.85)
train_gws = gws[:train_end]
val_gws = gws[train_end:val_end]
test_gws = gws[val_end:]

df_train_hi = df_fwd_train_hi[df_fwd_train_hi["gameweek"].isin(train_gws)].copy()
df_val_hi = df_fwd_train_hi[df_fwd_train_hi["gameweek"].isin(val_gws)].copy()
df_test_hi = df_fwd_train_hi[df_fwd_train_hi["gameweek"].isin(test_gws)].copy()

id_cols = ["player_id", "position", "name", "side", "gameweek"]
feature_cols_hi = [c for c in df_fwd_train_hi.columns if c not in id_cols + [target_col]]

X_train_hi, y_train_hi = df_train_hi[feature_cols_hi], df_train_hi[target_col]
X_val_hi, y_val_hi = df_val_hi[feature_cols_hi], df_val_hi[target_col]
X_test_hi, y_test_hi = df_test_hi[feature_cols_hi], df_test_hi[target_col]

print("Loaded:", src_path)
print("High-minute train rows total:", len(df_fwd_train_hi))
print("Train/Val/Test rows:", len(df_train_hi), len(df_val_hi), len(df_test_hi))
print("GW ranges:", (min(train_gws), max(train_gws)), (min(val_gws), max(val_gws)), (min(test_gws), max(test_gws)))
print("Features:", len(feature_cols_hi))


Loaded: /mnt/c/Users/easan.s.2024/Documents/GitHub/FPL-Oracle/data/intermediate/2025-2026_df_fwd_att.csv
High-minute train rows total: 548
Train/Val/Test rows: 390 81 77
GW ranges: (np.int64(6), np.int64(26)) (np.int64(27), np.int64(30)) (np.int64(31), np.int64(35))
Features: 70


/tmp/ipykernel_32902/2835244238.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{c}_sum_last5"] = lag.rolling(window=5, min_periods=5).sum()
/tmp/ipykernel_32902/2835244238.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{c}_sum_last3"] = lag.rolling(window=3, min_periods=3).sum()
/tmp/ipykernel_32902/2835244238.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.

In [13]:
# === TRAIN HIGH-MINUTES FORWARDS MODEL ===
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

model_hi = LGBMRegressor(
    objective="regression_l1",
    n_estimators=3000,
    learning_rate=0.02,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1,
)

model_hi.fit(
    X_train_hi,
    y_train_hi,
    eval_set=[(X_val_hi, y_val_hi)],
    eval_metric="l1",
    callbacks=[early_stopping(150, verbose=True), log_evaluation(100)],
)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

pred_val_hi = model_hi.predict(X_val_hi)
pred_test_hi = model_hi.predict(X_test_hi)

print("Best iteration:", getattr(model_hi, "best_iteration_", None))
print(
    "VAL  | MAE:",
    round(mean_absolute_error(y_val_hi, pred_val_hi), 4),
    "| RMSE:",
    round(rmse(y_val_hi, pred_val_hi), 4),
    "| R2:",
    round(r2_score(y_val_hi, pred_val_hi), 4),
)
print(
    "TEST | MAE:",
    round(mean_absolute_error(y_test_hi, pred_test_hi), 4),
    "| RMSE:",
    round(rmse(y_test_hi, pred_test_hi), 4),
    "| R2:",
    round(r2_score(y_test_hi, pred_test_hi), 4),
)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.069054 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4708
[LightGBM] [Info] Number of data points in the train set: 390, number of used features: 64
[LightGBM] [Info] Start training from score 2.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 150 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

In [14]:
# === TEST HIGH-MINUTES MODEL ===
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# Predict on held-out high-minutes test split
pred_test_hi = model_hi.predict(X_test_hi)

print(
    "TEST | MAE:", round(mean_absolute_error(y_test_hi, pred_test_hi), 4),
    "| RMSE:", round(rmse(y_test_hi, pred_test_hi), 4),
    "| R2:", round(r2_score(y_test_hi, pred_test_hi), 4)
)

# Show 5 random test rows (pred vs actual)
tmp = df_test_hi.reset_index(drop=True).copy()
yy = y_test_hi.reset_index(drop=True)
pp = pd.Series(pred_test_hi).reset_index(drop=True)

rng = np.random.default_rng(42)
idx = rng.choice(len(tmp), size=min(5, len(tmp)), replace=False)

out = tmp.loc[idx, ["player_id", "name", "position", "side", "gameweek"]].copy()
out["actual_points"] = yy.loc[idx].values
out["pred_points"] = pp.loc[idx].values
out["abs_error"] = (out["actual_points"] - out["pred_points"]).abs()

print("Sample predictions:")
print(out.sort_values("gameweek").to_string(index=False))


TEST | MAE: 2.4253 | RMSE: 4.0563 | R2: -0.1547
Sample predictions:
 player_id          name position side  gameweek  actual_points  pred_points  abs_error
       100   Bournemouth  Forward home        31            5.0         1.97       3.03
       661     Liverpool  Forward away        31            1.0         1.98       0.98
       526 Nott'm Forest  Forward away        35            5.0         2.00       3.00
       681       Man Utd  Forward home        35            5.0         2.06       2.94
       791      West Ham  Forward away        35            1.0         2.00       1.00
